# Stage 5: PEFT Adaptation Experiments

**Self-contained.** All code is inline.

**What this notebook does (in order):**
1. Defines all PEFT implementations (LoRA, Adapters, VPT, BitFit)
2. **Set 1:** Compares 5 PEFT methods at 100% labels (15 runs)
3. **Set 2:** Label efficiency sweep for top 2 methods (36 runs)
4. **Set 3:** Cross-domain evaluation on PlantDoc (no training)
5. **Set 4:** LoRA sensitivity sweep (12 runs)
6. Generates all key dissertation figures

**Estimated time on H200:** ~4-5 hours total

**Before running:** Update the `CONFIG` dict in Cell 1 with your actual paths.

## 1. Configuration & Imports

Update paths in the CONFIG dict below. Everything else should work as-is.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1: CONFIG — UPDATE THESE PATHS
# ═══════════════════════════════════════════════════════════════════════

from pathlib import Path

CONFIG = {
    # ── Dataset paths ──
    "PV_ROOT":              Path("/workspace/leaf-jepa/data/plantvillage_raw"),      
    "PD_ROOT":              Path("/workspace/leaf-jepa/data/plantdoc_raw"),         
    "SPLITS_DIR":           Path("/workspace/leaf-jepa/stage2_dataset_preparation/outputs/splits"),        
    "CLASS_WEIGHTS_PATH":   Path("/workspace/leaf-jepa/stage2_dataset_preparation/outputs/preprocessing/class_weights.json"), 

    # ── Checkpoints ──
    "IJEPA_CHECKPOINT":     Path("/workspace/leaf-jepa/stage3_baseline_establishment/checkpoints/IN1K-vit.h.14-300e.pth.tar"),  
    "LEAF_JEPA_CHECKPOINT": Path("/workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/leafjepa-vit-h14-best.pth"),           

    # ── Normalisation (from normalisation_stats.json) ──
    "NORM_MEAN" : [0.466726, 0.488969, 0.41028],
    "NORM_STD"  : [0.195034, 0.170282, 0.213409],

    # ── Output ──
    "OUTPUT_DIR":  Path("/workspace/leaf-jepa/stage5_peft_adaptation_experiments/outputs/peft"),
    "FIGURES_DIR": Path("/workspace/leaf-jepa/stage5_peft_adaptation_experiments/outputs/figures"),

    # ── Model ──
    "VIT_MODEL":   "vit_huge_patch14_224",
    "EMBED_DIM":   1280,
    "NUM_CLASSES":  38,
    "VIT_DEPTH":    32,

    # ── Training (shared across ALL methods) ──
    "BATCH_SIZE":   32,
    "MAX_EPOCHS":   5,
    "PATIENCE":     2,
    "WEIGHT_DECAY": 0.01,
    "WARMUP_FRAC":  0.10,
    "GRAD_CLIP":    1.0,
    "NUM_WORKERS":  4,
    "SEEDS":        [42, 123],
    "FRACTIONS":    [0.01, 0.05, 0.10, 0.25, 0.50, 1.00],

    # ── Per-method LR (reasonable defaults — skip LR sweep to save time) ──
    "LR": {
        "lora": 3e-4,
        "adapter": 3e-4,
        "vpt_shallow": 3e-4,
        "vpt_deep": 3e-4,
        "bitfit": 1e-3,
    },

    # ── WandB (set to None to disable) ──
    "WANDB_PROJECT": "leaf-jepa-irp",   # Set to "leaf-jepa-irp" to enable
    "WANDB_ENTITY":  "muh-haleef02-inform",   # Your username
}

# Create output dirs
CONFIG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)
CONFIG["FIGURES_DIR"].mkdir(parents=True, exist_ok=True)
(CONFIG["OUTPUT_DIR"] / "checkpoints").mkdir(parents=True, exist_ok=True)

# Verify
print("CONFIG VERIFICATION")
print("=" * 60)
for key in ["PV_ROOT", "PD_ROOT", "SPLITS_DIR", "IJEPA_CHECKPOINT", "LEAF_JEPA_CHECKPOINT"]:
    p = CONFIG[key]
    ok = p.exists() if p else False
    print(f"  {'OK' if ok else 'MISSING':>7}  {key}: {p}")
print("=" * 60)

CONFIG VERIFICATION
       OK  PV_ROOT: /workspace/leaf-jepa/data/plantvillage_raw
       OK  PD_ROOT: /workspace/leaf-jepa/data/plantdoc_raw
       OK  SPLITS_DIR: /workspace/leaf-jepa/stage2_dataset_preparation/outputs/splits
       OK  IJEPA_CHECKPOINT: /workspace/leaf-jepa/stage3_baseline_establishment/checkpoints/IN1K-vit.h.14-300e.pth.tar
       OK  LEAF_JEPA_CHECKPOINT: /workspace/leaf-jepa/stage4_leaf_jepa_pretraining/outputs/checkpoints/leafjepa-vit-h14-best.pth


## 2. Imports

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2: IMPORTS
# ═══════════════════════════════════════════════════════════════════════

import os, json, time, math, copy, random, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import timm

# AMP — compatible with PyTorch 1.x and 2.x
try:
    from torch.amp import GradScaler, autocast
except ImportError:
    from torch.cuda.amp import GradScaler, autocast

AMP_DEVICE = "cuda"

print(f"PyTorch: {torch.__version__}")
print(f"timm:    {timm.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.4.1+cu124
timm:    0.9.16
CUDA:    True
GPU:     NVIDIA H200
VRAM:    150.1 GB


## 3. Core Utilities (seed, data, train, eval)

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3: CORE UTILITIES
# ═══════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Dataset ──────────────────────────────────────────────────────────

class PVDataset(Dataset):
    """Loads from Stage 2 JSON splits: {paths: [...], labels: [...], class_names: [...]}"""
    def __init__(self, root, json_path, transform):
        self.root = Path(root)
        self.transform = transform
        with open(json_path) as f:
            data = json.load(f)
        self.paths = data["paths"]
        self.labels = data["labels"]  # already integer indices from Stage 2
        self.class_names = data.get("class_names", [])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.root / self.paths[idx]).convert("RGB")
        return self.transform(img), int(self.labels[idx])


class PDDataset(Dataset):
    def __init__(self, root, transform, class_filter=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        dirs = sorted([d for d in self.root.iterdir()
                       if d.is_dir() and (class_filter is None or d.name in class_filter)])
        self.class_names = [d.name for d in dirs]
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}
        for d in dirs:
            for f in d.iterdir():
                if f.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.samples.append((f, self.class_to_idx[d.name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), label

# ── Transforms ───────────────────────────────────────────────────────

def get_train_tf():
    return transforms.Compose([
        transforms.Resize(256), transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["NORM_MEAN"], CONFIG["NORM_STD"]),
    ])

def get_eval_tf():
    return transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["NORM_MEAN"], CONFIG["NORM_STD"]),
    ])

# ── Data loaders ─────────────────────────────────────────────────────

def get_loaders(fraction=1.0, seed=42, bs=None):
    bs = bs or CONFIG["BATCH_SIZE"]
    sd = CONFIG["SPLITS_DIR"]
    pv = CONFIG["PV_ROOT"]
    nw = CONFIG["NUM_WORKERS"]

    # Stage 2 saves: train_split.json, val_split.json, test_split.json
    # Fractions: SPLITS_DIR/fractions/fraction_0.10_seed42.json
    if fraction < 1.0:
        frac_dir = sd / "fractions"
        train_json = frac_dir / f"fraction_{fraction:.2f}_seed{seed}.json"
    else:
        train_json = sd / "train_split.json"

    assert train_json.exists(), f"Split not found: {train_json}"

    train_ds = PVDataset(pv, train_json, get_train_tf())
    val_ds   = PVDataset(pv, sd / "val_split.json", get_eval_tf())
    test_ds  = PVDataset(pv, sd / "test_split.json", get_eval_tf())

    kw = dict(num_workers=nw, pin_memory=True)
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=False, **kw),
            DataLoader(val_ds, batch_size=bs, shuffle=False, **kw),
            DataLoader(test_ds, batch_size=bs, shuffle=False, **kw),
            len(train_ds))

# ── Encoder loading ──────────────────────────────────────────────────

def load_encoder(ckpt_path, freeze=True):
    enc = timm.create_model(CONFIG["VIT_MODEL"], pretrained=False,
                            num_classes=0, global_pool="", no_embed_class=True)
    ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
    sd = ckpt.get("target_encoder", ckpt.get("encoder", ckpt.get("model", ckpt)))
    sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in sd.items()}
    enc.load_state_dict(sd, strict=False)
    if freeze:
        for p in enc.parameters():
            p.requires_grad = False
        enc.eval()
    return enc.to(device)

print("Core utilities defined.")

Core utilities defined.


## 4. Training Engine

In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: TRAINING ENGINE
# ═══════════════════════════════════════════════════════════════════════

class CosineWarmup:
    def __init__(self, optim, warmup, total):
        self.optim = optim
        self.warmup = warmup
        self.total = total
        self.base_lrs = [g["lr"] for g in optim.param_groups]
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup:
            s = self.step_count / max(1, self.warmup)
        else:
            p = (self.step_count - self.warmup) / max(1, self.total - self.warmup)
            s = 0.5 * (1 + math.cos(math.pi * p))
        for g, blr in zip(self.optim.param_groups, self.base_lrs):
            g["lr"] = blr * s


class EarlyStop:
    def __init__(self, patience=10):
        self.patience = patience
        self.best = None
        self.best_epoch = 0
        self.wait = 0
        self.stop = False

    def __call__(self, score, epoch):
        if self.best is None or score > self.best + 1e-4:
            self.best = score
            self.best_epoch = epoch
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stop = True
        return self.stop


def train_epoch(model, loader, optim, crit, scaler, sched):
    model.train()
    tot_loss = correct = total = 0
    for imgs, labs in loader:
        imgs, labs = imgs.to(device), labs.to(device)
        optim.zero_grad(set_to_none=True)
        with autocast(AMP_DEVICE):
            logits = model(imgs)
            if logits.dim() == 3:
                    logits = logits[:, 0, :]
            loss = crit(logits, labs)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["GRAD_CLIP"])
        scaler.step(optim)
        scaler.update()
        if sched:
            sched.step()
        tot_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labs).sum().item()
        total += labs.size(0)
    return {"loss": tot_loss/total, "acc": correct/total}


@torch.no_grad()
def eval_model(model, loader, crit):
    model.eval()
    tot_loss = total = 0
    preds_all, labs_all = [], []
    for imgs, labs in loader:
        imgs, labs = imgs.to(device), labs.to(device)
        with autocast(AMP_DEVICE):
            logits = model(imgs)
            if logits.dim() == 3:
                    logits = logits[:, 0, :]
            loss = crit(logits, labs)
        tot_loss += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labs_all.extend(labs.cpu().numpy())
        total += labs.size(0)
    p, l = np.array(preds_all), np.array(labs_all)
    nc = CONFIG["NUM_CLASSES"]
    return {
        "loss": tot_loss/total,
        "acc": accuracy_score(l, p),
        "macro_f1": f1_score(l, p, average="macro", zero_division=0),
        "per_class_f1": f1_score(l, p, average=None, zero_division=0).tolist(),
        "cm": confusion_matrix(l, p, labels=list(range(nc))),
    }


def run_training(model, fraction=1.0, seed=42, lr=3e-4, tag=""):
    """Full training pipeline. Returns result dict."""
    set_seed(seed)
    t0 = time.time()

    train_loader, val_loader, test_loader, n_train = get_loaders(fraction, seed)

    # Class weights
    cwp = CONFIG["CLASS_WEIGHTS_PATH"]
    if cwp and Path(cwp).exists():
        w = json.loads(Path(cwp).read_text())
        if isinstance(w, dict):
            wt = torch.zeros(CONFIG["NUM_CLASSES"])
            for k, v in w.items():
                wt[int(k)] = v
        else:
            wt = torch.tensor(w, dtype=torch.float32)
        crit = nn.CrossEntropyLoss(weight=wt.to(device))
    else:
        crit = nn.CrossEntropyLoss()

    trainable = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    n_total = sum(p.numel() for p in model.parameters())

    optim = torch.optim.AdamW(trainable, lr=lr, weight_decay=CONFIG["WEIGHT_DECAY"])
    total_steps = CONFIG["MAX_EPOCHS"] * len(train_loader)
    sched = CosineWarmup(optim, int(total_steps * CONFIG["WARMUP_FRAC"]), total_steps)
    scaler = GradScaler()
    es = EarlyStop(CONFIG["PATIENCE"])
    best_state = None

    torch.cuda.reset_peak_memory_stats()
    epoch_times = []

    for ep in range(1, CONFIG["MAX_EPOCHS"] + 1):
        et0 = time.time()
        tm = train_epoch(model, train_loader, optim, crit, scaler, sched)
        vm = eval_model(model, val_loader, crit)
        epoch_times.append(time.time() - et0)

        if ep % 5 == 0 or ep == 1:
            print(f"    Ep {ep:3d} | TrL {tm['loss']:.4f} | VF1 {vm['macro_f1']:.4f} | "
                  f"VAcc {vm['acc']:.4f} | {epoch_times[-1]:.1f}s")

        if es(vm["macro_f1"], ep):
            print(f"    Early stop ep {ep} (best={es.best:.4f} @{es.best_epoch})")
            break
        if es.best_epoch == ep:
            best_state = copy.deepcopy(model.state_dict())

    if best_state:
        model.load_state_dict(best_state)
    test = eval_model(model, test_loader, crit)

    vram = torch.cuda.max_memory_allocated() / 1e6
    total_time = time.time() - t0

    result = {
        "tag": tag,
        "fraction": fraction,
        "seed": seed,
        "lr": lr,
        "trainable_params": n_trainable,
        "total_params": n_total,
        "pct_trainable": round(100 * n_trainable / n_total, 4),
        "test_macro_f1": test["macro_f1"],
        "test_accuracy": test["acc"],
        "val_macro_f1": es.best,
        "best_epoch": es.best_epoch,
        "peak_vram_mb": round(vram, 1),
        "avg_epoch_time_s": round(np.mean(epoch_times[1:]) if len(epoch_times) > 1 else epoch_times[0], 2),
        "total_time_s": round(total_time, 1),
        "per_class_f1": test["per_class_f1"],
    }

    print(f"  ★ TEST: F1={test['macro_f1']:.4f} Acc={test['acc']:.4f} | "
          f"Params={n_trainable:,} ({result['pct_trainable']:.3f}%) | "
          f"VRAM={vram:.0f}MB | {total_time:.0f}s")

    return result

print("Training engine defined.")

Training engine defined.


## 5. PEFT Implementations (all inline)

In [13]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5: PEFT IMPLEMENTATIONS
# ═══════════════════════════════════════════════════════════════════════

# ── LoRA ──────────────────────────────────────────────────────────────

class LoRAQKV(nn.Module):
    """LoRA for timm's fused QKV. Adds to Q and V only."""
    def __init__(self, orig_qkv, rank, alpha=None, dropout=0.0):
        super().__init__()
        self.orig = orig_qkv
        orig_qkv.weight.requires_grad = False
        if orig_qkv.bias is not None:
            orig_qkv.bias.requires_grad = False

        d_in = orig_qkv.in_features
        d_head = orig_qkv.out_features // 3
        self.rank = rank
        self.scale = (alpha or float(rank)) / rank
        self.d_head = d_head

        self.A_q = nn.Parameter(torch.zeros(rank, d_in))
        self.B_q = nn.Parameter(torch.zeros(d_head, rank))
        self.A_v = nn.Parameter(torch.zeros(rank, d_in))
        self.B_v = nn.Parameter(torch.zeros(d_head, rank))
        nn.init.kaiming_uniform_(self.A_q, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.A_v, a=math.sqrt(5))
        nn.init.zeros_(self.B_q)
        nn.init.zeros_(self.B_v)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        qkv = self.orig(x)
        xd = self.drop(x)
        dq = (xd @ self.A_q.T @ self.B_q.T) * self.scale
        dv = (xd @ self.A_v.T @ self.B_v.T) * self.scale
        qkv = qkv.clone()
        qkv[:, :, :self.d_head] += dq
        qkv[:, :, 2*self.d_head:] += dv
        return qkv


def apply_lora(encoder, rank=8, target_blocks=None):
    for i, blk in enumerate(encoder.blocks):
        if target_blocks is not None and i not in target_blocks:
            continue
        blk.attn.qkv = LoRAQKV(blk.attn.qkv, rank)

# ── Adapters ──────────────────────────────────────────────────────────

class Adapter(nn.Module):
    def __init__(self, dim, bottleneck):
        super().__init__()
        self.ln = nn.LayerNorm(dim)
        self.down = nn.Linear(dim, bottleneck)
        self.up = nn.Linear(bottleneck, dim)
        nn.init.normal_(self.down.weight, std=1e-3)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(self.ln(x))))


class AdaptedBlock(nn.Module):
    def __init__(self, block, dim, bottleneck):
        super().__init__()
        self.block = block
        self.ada_attn = Adapter(dim, bottleneck)
        self.ada_ffn = Adapter(dim, bottleneck)
        for p in self.block.parameters():
            p.requires_grad = False

    def forward(self, x):
        # MHA
        a = self.block.attn(self.block.norm1(x))
        dp1 = getattr(self.block, "drop_path1", None)
        ls1 = getattr(self.block, "ls1", None)
        if ls1: a = ls1(a)
        if dp1: a = dp1(a)
        x = x + a
        x = self.ada_attn(x)
        # FFN
        f = self.block.mlp(self.block.norm2(x))
        dp2 = getattr(self.block, "drop_path2", None)
        ls2 = getattr(self.block, "ls2", None)
        if ls2: f = ls2(f)
        if dp2: f = dp2(f)
        x = x + f
        x = self.ada_ffn(x)
        return x


def apply_adapters(encoder, bottleneck=64):
    new_blocks = nn.Sequential()
    for blk in encoder.blocks:
        new_blocks.append(AdaptedBlock(blk, CONFIG["EMBED_DIM"], bottleneck))
    encoder.blocks = new_blocks

# ── VPT Shallow ───────────────────────────────────────────────────────

class VPTShallow(nn.Module):
    def __init__(self, encoder, n_prompts=50):
        super().__init__()
        self.enc = encoder
        self.n_prompts = n_prompts
        self.prompts = nn.Parameter(torch.zeros(1, n_prompts, CONFIG["EMBED_DIM"]))
        nn.init.normal_(self.prompts, std=0.02)
        for p in self.enc.parameters():
            p.requires_grad = False

    def forward(self, x):
        B = x.shape[0]
        x = self.enc.patch_embed(x)
        # I-JEPA: no CLS token, pos_embed matches patch count directly
        x = self.enc.pos_drop(x + self.enc.pos_embed)
        x = torch.cat([self.prompts.expand(B,-1,-1), x], dim=1)
        x = self.enc.blocks(x)
        x = self.enc.norm(x)
        x = x[:, self.n_prompts:, :]  # remove prompts
        x = x.mean(1)  # global avg pool over patches
        return x

# ── VPT Deep ─────────────────────────────────────────────────────────

class VPTDeep(nn.Module):
    def __init__(self, encoder, n_prompts=50):
        super().__init__()
        self.enc = encoder
        self.n_prompts = n_prompts
        self.prompts = nn.ParameterList([
            nn.Parameter(torch.zeros(1, n_prompts, CONFIG["EMBED_DIM"]))
            for _ in range(len(encoder.blocks))
        ])
        for pt in self.prompts:
            nn.init.normal_(pt, std=0.02)
        for p in self.enc.parameters():
            p.requires_grad = False

    def forward(self, x):
        B = x.shape[0]
        x = self.enc.patch_embed(x)
        x = self.enc.pos_drop(x + self.enc.pos_embed)
        for i, blk in enumerate(self.enc.blocks):
            x = torch.cat([self.prompts[i].expand(B,-1,-1), x], dim=1)
            x = blk(x)
            x = x[:, self.n_prompts:, :]
        x = self.enc.norm(x)
        x = x.mean(1)
        return x

# ── Model builder ────────────────────────────────────────────────────

class Classifier(nn.Module):
    def __init__(self, encoder, dim, n_cls):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(dim, n_cls)

    def forward(self, x):
        return self.head(self.encoder(x))


def build_model(method, ckpt_path, rank=8, bottleneck=64, n_prompts=50, target_blocks=None):
    """Build a PEFT classifier. Returns model on device."""
    enc = load_encoder(ckpt_path, freeze=True)
    dim = CONFIG["EMBED_DIM"]
    nc = CONFIG["NUM_CLASSES"]

    if method == "lora":
        apply_lora(enc, rank=rank, target_blocks=target_blocks)
        model = Classifier(enc, dim, nc)
    elif method == "adapter":
        apply_adapters(enc, bottleneck=bottleneck)
        model = Classifier(enc, dim, nc)
    elif method == "vpt_shallow":
        wrapped = VPTShallow(enc, n_prompts)
        model = Classifier(wrapped, dim, nc)
    elif method == "vpt_deep":
        wrapped = VPTDeep(enc, n_prompts)
        model = Classifier(wrapped, dim, nc)
    elif method == "bitfit":
        model = Classifier(enc, dim, nc)
        for n, p in model.encoder.named_parameters():
            if "bias" in n:
                p.requires_grad = True
    elif method == "linear_probe":
        model = Classifier(enc, dim, nc)
    else:
        raise ValueError(f"Unknown method: {method}")

    # Head always trainable
    for p in model.head.parameters():
        p.requires_grad = True

    model = model.to(device)
    tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  {method}: {tp:,} trainable / {total:,} total ({100*tp/total:.4f}%)")
    return model

print("PEFT implementations defined.")

PEFT implementations defined.


## 6. SET 1: Method Comparison at 100% Labels

Compares all 5 PEFT methods at their default best HP.
**15 runs:** 5 methods × 3 seeds.

This is reduced from the full 36-run grid — we skip the HP sweep within each method
and use one strong configuration per method. This is sufficient to answer RQ2.

In [16]:
set1_methods = [
    # ("lora",        {"rank": 8}),
    # ("adapter",     {"bottleneck": 64}),
    # ("vpt_shallow", {"n_prompts": 50}),
    # ("bitfit",      {}),
    ("vpt_deep",    {"n_prompts": 50}),
]

set1_results = []
total_runs = len(set1_methods) * len(CONFIG["SEEDS"])
run_times = []

for i, (method, kwargs) in enumerate(set1_methods):
    for j, seed in enumerate(CONFIG["SEEDS"]):
        run_num = i * len(CONFIG["SEEDS"]) + j + 1
        tag = f"S5-{method}-frac1.00-seed{seed}"
        
        eta = f" | ETA: {np.mean(run_times) * (total_runs - run_num + 1) / 60:.0f}min" if run_times else ""
        print(f"\n{'─'*60}")
        print(f"  [{run_num}/{total_runs}] {tag}{eta}")
        print(f"{'─'*60}")
        
        t0 = time.time()
        model = build_model(method, CONFIG["LEAF_JEPA_CHECKPOINT"], **kwargs)
        result = run_training(
            model, fraction=1.0, seed=seed,
            lr=CONFIG["LR"][method], tag=tag,
        )
        run_times.append(time.time() - t0)
        
        result["method"] = method
        result["hp"] = kwargs
        set1_results.append(result)
        del model
        torch.cuda.empty_cache()
        
        # Save after EVERY run (crash protection)
        save_path = CONFIG["OUTPUT_DIR"] / "set1_method_comparison.json"
        with open(save_path, "w") as f:
            json.dump(set1_results, f, indent=2)

print(f"\n{'='*60}")
print(f"  Set 1 COMPLETE: {total_runs} runs in {sum(run_times)/60:.1f} min")
print(f"  Avg per run: {np.mean(run_times):.0f}s")
print(f"  Saved: {save_path}")
print(f"{'='*60}")


────────────────────────────────────────────────────────────
  [1/2] S5-vpt_deep-frac1.00-seed42
────────────────────────────────────────────────────────────
  vpt_deep: 2,096,678 trainable / 632,860,198 total (0.3313%)
    Ep   1 | TrL 1.3677 | VF1 0.9050 | VAcc 0.9413 | 308.8s
    Ep   5 | TrL 0.0319 | VF1 0.9862 | VAcc 0.9891 | 305.1s
    Ep  10 | TrL 0.0077 | VF1 0.9941 | VAcc 0.9962 | 313.6s
  ★ TEST: F1=0.9945 Acc=0.9963 | Params=2,096,678 (0.331%) | VRAM=29582MB | 3071s

────────────────────────────────────────────────────────────
  [2/2] S5-vpt_deep-frac1.00-seed123 | ETA: 52min
────────────────────────────────────────────────────────────
  vpt_deep: 2,096,678 trainable / 632,860,198 total (0.3313%)


KeyboardInterrupt: 

In [17]:
import json
from pathlib import Path

set1_results = []
for f in sorted(CONFIG["OUTPUT_DIR"].glob("set1_method_comparison*.json")):
    data = json.loads(f.read_text())
    if isinstance(data, list):
        set1_results.extend(data)
    else:
        set1_results.append(data)

print(f"Loaded {len(set1_results)} results from {len(list(CONFIG['OUTPUT_DIR'].glob('set1_method_comparison*.json')))} files")
for r in set1_results:
    print(f"  {r['method']}: F1={r['test_macro_f1']:.4f} | Params={r['trainable_params']:,}")

Loaded 6 results from 5 files
  bitfit: F1=0.9939 | Params=501,798
  vpt_shallow: F1=0.9705 | Params=112,678
  adapter: F1=0.9967 | Params=10,784,294
  lora: F1=0.9960 | Params=1,359,398
  lora: F1=0.9950 | Params=1,359,398
  vpt_deep: F1=0.9945 | Params=2,096,678


In [18]:
method_scores = defaultdict(list)
method_params = {}
for r in set1_results:
    m = r["method"]
    method_scores[m].append(r["test_macro_f1"])
    method_params[m] = r["trainable_params"]

ranked = []
for m, scores in method_scores.items():
    std = np.std(scores) if len(scores) > 1 else 0.0
    ranked.append((m, np.mean(scores), std, method_params[m]))
ranked.sort(key=lambda x: x[1], reverse=True)

for i, (m, mean, std, params) in enumerate(ranked):
    marker = " ◀ BEST" if i == 0 else ""
    print(f"  {i+1}. {m:<15} | F1: {mean:.4f} ± {std:.4f} | Params: {params:>10,}{marker}")

TOP2 = [ranked[0][0], ranked[1][0]]
print(f"\n  ★ Top 2 for label efficiency sweep: {TOP2}")

  1. adapter         | F1: 0.9967 ± 0.0000 | Params: 10,784,294 ◀ BEST
  2. lora            | F1: 0.9955 ± 0.0005 | Params:  1,359,398
  3. vpt_deep        | F1: 0.9945 ± 0.0000 | Params:  2,096,678
  4. bitfit          | F1: 0.9939 ± 0.0000 | Params:    501,798
  5. vpt_shallow     | F1: 0.9705 ± 0.0000 | Params:    112,678

  ★ Top 2 for label efficiency sweep: ['adapter', 'lora']


## 6b. Set 1 Summary

In [19]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6b: SET 1 SUMMARY
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("  SET 1 RESULTS: METHOD COMPARISON AT 100% LABELS")
print("=" * 80)

method_scores = defaultdict(list)
method_params = {}
for r in set1_results:
    m = r["method"]
    method_scores[m].append(r["test_macro_f1"])
    method_params[m] = r["trainable_params"]

ranked = []
for m, scores in method_scores.items():
    ranked.append((m, np.mean(scores), np.std(scores), method_params[m]))
ranked.sort(key=lambda x: x[1], reverse=True)

for i, (m, mean, std, params) in enumerate(ranked):
    marker = " ◀ BEST" if i == 0 else ""
    print(f"  {i+1}. {m:<15} | F1: {mean:.4f} ± {std:.4f} | Params: {params:>10,}{marker}")

TOP2 = [ranked[0][0], ranked[1][0]]
print(f"\n  ★ Top 2 for label efficiency sweep: {TOP2}")


  SET 1 RESULTS: METHOD COMPARISON AT 100% LABELS
  1. adapter         | F1: 0.9967 ± 0.0000 | Params: 10,784,294 ◀ BEST
  2. lora            | F1: 0.9955 ± 0.0005 | Params:  1,359,398
  3. vpt_deep        | F1: 0.9945 ± 0.0000 | Params:  2,096,678
  4. bitfit          | F1: 0.9939 ± 0.0000 | Params:    501,798
  5. vpt_shallow     | F1: 0.9705 ± 0.0000 | Params:    112,678

  ★ Top 2 for label efficiency sweep: ['adapter', 'lora']


## 7. SET 2: Label Efficiency Sweep

Top 2 methods × 6 fractions × 3 seeds = **36 runs**.
This answers RQ3 and RQ5.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7: SET 2 — LABEL EFFICIENCY
# ═══════════════════════════════════════════════════════════════════════

# Get HP for top methods from Set 1
method_hp = {r["method"]: r["hp"] for r in set1_results}

set2_results = []

for method in TOP2:
    hp = method_hp.get(method, {})

    for frac in CONFIG["FRACTIONS"]:
        for seed in CONFIG["SEEDS"]:
            tag = f"S5-{method}-frac{frac:.2f}-seed{seed}"
            print(f"\n{'─'*50}\n  {tag}\n{'─'*50}")

            model = build_model(method, CONFIG["LEAF_JEPA_CHECKPOINT"], **hp)
            result = run_training(
                model, fraction=frac, seed=seed,
                lr=CONFIG["LR"][method], tag=tag,
            )
            result["method"] = method
            result["hp"] = hp
            set2_results.append(result)

            del model
            torch.cuda.empty_cache()

save_path = CONFIG["OUTPUT_DIR"] / "set2_label_efficiency.json"
with open(save_path, "w") as f:
    json.dump(set2_results, f, indent=2)
print(f"\nSet 2 saved: {save_path}")


──────────────────────────────────────────────────
  S5-adapter-frac0.01-seed42
──────────────────────────────────────────────────
  adapter: 10,784,294 trainable / 641,547,814 total (1.6810%)
    Ep   1 | TrL 3.7070 | VF1 0.0356 | VAcc 0.2134 | 37.0s
    Ep   5 | TrL 1.3195 | VF1 0.4327 | VAcc 0.6310 | 26.3s
    Ep  10 | TrL 0.4790 | VF1 0.6393 | VAcc 0.7741 | 40.5s
  ★ TEST: F1=0.6460 Acc=0.7707 | Params=10,784,294 (1.681%) | VRAM=21685MB | 426s

──────────────────────────────────────────────────
  S5-adapter-frac0.01-seed123
──────────────────────────────────────────────────
  adapter: 10,784,294 trainable / 641,547,814 total (1.6810%)
    Ep   1 | TrL 3.6660 | VF1 0.0337 | VAcc 0.2012 | 43.8s
    Ep   5 | TrL 1.3481 | VF1 0.4224 | VAcc 0.6460 | 37.2s
    Ep  10 | TrL 0.4819 | VF1 0.6378 | VAcc 0.7704 | 39.4s
  ★ TEST: F1=0.6465 Acc=0.7787 | Params=10,784,294 (1.681%) | VRAM=21641MB | 448s

──────────────────────────────────────────────────
  S5-adapter-frac0.05-seed42
────────────

## 7b. Label Efficiency Curves (KEY FIGURE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7b: LABEL EFFICIENCY PLOT
# ═══════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(10, 6))

# Group Set 2 results
grouped = defaultdict(lambda: defaultdict(list))
for r in set2_results:
    grouped[r["method"]][r["fraction"]].append(r["test_macro_f1"])

colours = {"lora": "#e74c3c", "adapter": "#3498db", "vpt_shallow": "#2ecc71",
           "vpt_deep": "#9b59b6", "bitfit": "#f39c12"}

for method, frac_data in grouped.items():
    fracs = sorted(frac_data.keys())
    means = [np.mean(frac_data[f]) for f in fracs]
    stds = [np.std(frac_data[f]) for f in fracs]
    c = colours.get(method, "#7f8c8d")
    ax.errorbar(fracs, means, yerr=stds, fmt="o-", color=c, label=f"Leaf-JEPA + {method}",
                capsize=4, linewidth=2, markersize=6)

# TODO: Add B1/B2 baselines from Stage 3 results here
# b1_fracs = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
# b1_means = [...]; b1_stds = [...]
# ax.errorbar(b1_fracs, b1_means, yerr=b1_stds, fmt="s--", color="#95a5a6", label="ResNet-50", ...)

ax.set_xlabel("Label Fraction", fontsize=12)
ax.set_ylabel("Macro-F1", fontsize=12)
ax.set_title("Label Efficiency: Leaf-JEPA + PEFT vs Baselines", fontsize=14)
ax.set_xscale("log")
ax.set_xticks([0.01, 0.05, 0.10, 0.25, 0.50, 1.00])
ax.get_xaxis().set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG["FIGURES_DIR"] / "label_efficiency_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {CONFIG['FIGURES_DIR'] / 'label_efficiency_curves.png'}")

## 8. SET 3: Cross-Domain Transfer (PlantDoc)

No training on PlantDoc. Extract features + k-NN evaluation.
Answers RQ4.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8: SET 3 — CROSS-DOMAIN
# ═══════════════════════════════════════════════════════════════════════

set_seed(42)

eval_tf = get_eval_tf()

# Build loaders
pv_train_ds = PVDataset(CONFIG["PV_ROOT"], CONFIG["SPLITS_DIR"] / "train.csv", eval_tf)
pv_loader = DataLoader(pv_train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                       num_workers=CONFIG["NUM_WORKERS"], pin_memory=True)

# PlantDoc — filter to classes with >=10 samples
pd_class_counts = {}
for d in CONFIG["PD_ROOT"].iterdir():
    if d.is_dir():
        n = len([f for f in d.iterdir() if f.suffix.lower() in {".jpg",".jpeg",".png"}])
        if n >= 10:
            pd_class_counts[d.name] = n

pd_classes = list(pd_class_counts.keys())
print(f"PlantDoc eval classes (>=10 samples): {len(pd_classes)}")

pd_ds = PDDataset(CONFIG["PD_ROOT"], eval_tf, class_filter=pd_classes)
pd_loader = DataLoader(pd_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False,
                       num_workers=CONFIG["NUM_WORKERS"], pin_memory=True)
print(f"PV train: {len(pv_train_ds)} | PD eval: {len(pd_ds)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8b: FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def extract_feats(encoder, loader):
    encoder.eval()
    feats, labs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast(AMP_DEVICE):
            f = encoder(imgs)
        feats.append(f.float().cpu().numpy())
        labs.append(labels.numpy())
    return np.concatenate(feats), np.concatenate(labs)


def run_knn(tr_f, tr_l, te_f, te_l, ks=[5, 10, 20]):
    tr_n = normalize(tr_f)
    te_n = normalize(te_f)
    results = {}
    for k in ks:
        knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", algorithm="brute")
        knn.fit(tr_n, tr_l)
        preds = knn.predict(te_n)
        mf1 = f1_score(te_l, preds, average="macro", zero_division=0)
        results[k] = mf1
        print(f"  k={k:3d} | F1={mf1:.4f}")
    return results


# Generic I-JEPA
print("\nGeneric I-JEPA features:")
gen_enc = load_encoder(CONFIG["IJEPA_CHECKPOINT"])
gen_pv_f, gen_pv_l = extract_feats(gen_enc, pv_loader)
gen_pd_f, gen_pd_l = extract_feats(gen_enc, pd_loader)
del gen_enc; torch.cuda.empty_cache()
gen_knn = run_knn(gen_pv_f, gen_pv_l, gen_pd_f, gen_pd_l)

# Leaf-JEPA
print("\nLeaf-JEPA features:")
leaf_enc = load_encoder(CONFIG["LEAF_JEPA_CHECKPOINT"])
leaf_pv_f, leaf_pv_l = extract_feats(leaf_enc, pv_loader)
leaf_pd_f, leaf_pd_l = extract_feats(leaf_enc, pd_loader)
del leaf_enc; torch.cuda.empty_cache()
leaf_knn = run_knn(leaf_pv_f, leaf_pv_l, leaf_pd_f, leaf_pd_l)

# RQ4 answer
best_k = max(gen_knn, key=gen_knn.get)
g_f1 = gen_knn[best_k]
l_f1 = leaf_knn[best_k]
delta = l_f1 - g_f1

print(f"\n{'='*50}")
print(f"  RQ4: Cross-domain transfer (k={best_k})")
print(f"  Generic I-JEPA: {g_f1:.4f}")
print(f"  Leaf-JEPA:      {l_f1:.4f}")
print(f"  Delta:          {delta:+.4f}")
print(f"{'='*50}")

cd_results = {"generic_knn": gen_knn, "leaf_knn": leaf_knn, "delta": delta}
with open(CONFIG["OUTPUT_DIR"] / "set3_cross_domain.json", "w") as f:
    json.dump(cd_results, f, indent=2)

## 8c. Domain Gap t-SNE

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8c: t-SNE DOMAIN GAP
# ═══════════════════════════════════════════════════════════════════════
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, pv_f, pd_f) in zip(axes, [
    ("Generic I-JEPA", gen_pv_f, gen_pd_f),
    ("Leaf-JEPA", leaf_pv_f, leaf_pd_f),
]):
    rng = np.random.RandomState(42)
    npv = min(2000, len(pv_f))
    npd = min(1000, len(pd_f))
    idx_pv = rng.choice(len(pv_f), npv, replace=False)
    idx_pd = rng.choice(len(pd_f), npd, replace=False)

    combined = np.concatenate([pv_f[idx_pv], pd_f[idx_pd]])
    labels = ["PV"]*npv + ["PD"]*npd

    emb = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000).fit_transform(combined)

    for ds, c, m in [("PV","#3498db","o"), ("PD","#e74c3c","^")]:
        mask = np.array(labels) == ds
        ax.scatter(emb[mask,0], emb[mask,1], c=c, marker=m, s=12, alpha=0.4, label=ds)

    ax.set_title(name, fontsize=13)
    ax.legend()
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Domain Gap: PlantVillage vs PlantDoc", fontsize=15)
plt.tight_layout()
plt.savefig(CONFIG["FIGURES_DIR"] / "domain_gap_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9: SET 4 — LoRA SENSITIVITY
# ═══════════════════════════════════════════════════════════════════════

set4_ranks = [2, 4, 8, 16]
set4_results = []

for rank in set4_ranks:
    for seed in CONFIG["SEEDS"]:
        tag = f"S5-LoRA-r{rank}-frac1.00-seed{seed}"
        print(f"\n  {tag}")
        model = build_model("lora", CONFIG["LEAF_JEPA_CHECKPOINT"], rank=rank)
        result = run_training(model, fraction=1.0, seed=seed, lr=CONFIG["LR"]["lora"], tag=tag)
        result["method"] = "lora"
        result["hp"] = {"rank": rank}
        set4_results.append(result)
        del model; torch.cuda.empty_cache()

with open(CONFIG["OUTPUT_DIR"] / "set4_lora_sensitivity.json", "w") as f:
    json.dump(set4_results, f, indent=2)

# Pareto plot
fig, ax = plt.subplots(figsize=(9, 5))
for r in set4_results:
    ax.scatter(r["trainable_params"], r["test_macro_f1"], c="#e74c3c", s=60, alpha=0.7)
# Add Set 1 methods
for r in set1_results:
    c = {"lora":"#e74c3c","adapter":"#3498db","vpt_shallow":"#2ecc71",
         "vpt_deep":"#9b59b6","bitfit":"#f39c12"}.get(r["method"],"gray")
    ax.scatter(r["trainable_params"], r["test_macro_f1"], c=c, s=60, alpha=0.7, label=r["method"])

handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels,handles)).values(), dict(zip(labels,handles)).keys(), fontsize=9)
ax.set_xlabel("Trainable Parameters"); ax.set_ylabel("Macro-F1")
ax.set_xscale("log"); ax.set_title("Pareto: F1 vs Parameters")
ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(CONFIG["FIGURES_DIR"] / "pareto_frontier.png", dpi=150)
plt.show()

## 10. Final Summary & Export

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10: FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("  STAGE 5 COMPLETE — FINAL SUMMARY")
print("=" * 80)

# Set 1 summary
print("\n  SET 1: METHOD COMPARISON (100% labels)")
print("  " + "-" * 55)
for m, mean, std, params in ranked:
    print(f"    {m:<15} | F1={mean:.4f}±{std:.4f} | Params={params:,}")

# Set 2 summary
print("\n  SET 2: LABEL EFFICIENCY")
print("  " + "-" * 55)
for method, frac_data in grouped.items():
    print(f"    {method}:")
    for frac in sorted(frac_data.keys()):
        scores = frac_data[frac]
        print(f"      {frac:6.0%} → F1={np.mean(scores):.4f}±{np.std(scores):.4f}")

# Set 3 summary
print(f"\n  SET 3: CROSS-DOMAIN (RQ4)")
print(f"  " + "-" * 55)
print(f"    Generic I-JEPA → PlantDoc: {g_f1:.4f}")
print(f"    Leaf-JEPA → PlantDoc:      {l_f1:.4f}")
print(f"    Delta:                     {delta:+.4f}")

# Master results for Stage 7
master = {
    "set1": set1_results,
    "set2": set2_results,
    "set3": cd_results,
    "set4": set4_results if 'set4_results' in dir() else [],
}
with open(CONFIG["OUTPUT_DIR"] / "stage5_master_results.json", "w") as f:
    json.dump(master, f, indent=2, default=lambda x: x.tolist() if hasattr(x, "tolist") else str(x))

# Summary CSV
rows = []
for r in set1_results:
    rows.append({"method": r["method"], "fraction": 1.0, "seed": r["seed"],
                 "macro_f1": r["test_macro_f1"], "accuracy": r["test_accuracy"],
                 "trainable_params": r["trainable_params"], "pct_trainable": r["pct_trainable"],
                 "peak_vram_mb": r["peak_vram_mb"], "avg_epoch_time_s": r["avg_epoch_time_s"]})
pd.DataFrame(rows).to_csv(CONFIG["OUTPUT_DIR"] / "peft_summary.csv", index=False)

print(f"\n  All results saved to: {CONFIG['OUTPUT_DIR']}")
print(f"  Figures saved to:     {CONFIG['FIGURES_DIR']}")
print(f"\n  Files produced:")
for f in sorted(CONFIG["OUTPUT_DIR"].glob("*.json")):
    print(f"    {f.name}")
for f in sorted(CONFIG["FIGURES_DIR"].glob("*.png")):
    print(f"    {f.name}")

print("\n  🎉 Stage 5 DONE.")